# HR Employee Handbook RAG System

This notebook:
1. Loads `employee_handbook_print_1.pdf`
2. Chunks the text into manageable passages
3. Creates OpenAI embeddings for each chunk
4. Stores them in a **new** Supabase table `hr_documents`
5. Demonstrates semantic search over the handbook

## 0. Prerequisites

### Supabase SQL — run once in your project's SQL editor

```sql
-- Enable the pgvector extension (if not already enabled)
create extension if not exists vector;

-- New table dedicated to the HR handbook
create table if not exists hr_documents (
  id          bigserial primary key,
  content     text        not null,
  page_number int,
  chunk_index int,
  embedding   vector(1536)
);

-- Similarity-search function for this table
create or replace function match_hr_documents (
  query_embedding vector(1536),
  match_threshold float,
  match_count     int
)
returns table (
  id          bigint,
  content     text,
  page_number int,
  chunk_index int,
  similarity  float
)
language sql stable
as $$
  select
    id,
    content,
    page_number,
    chunk_index,
    1 - (embedding <=> query_embedding) as similarity
  from hr_documents
  where 1 - (embedding <=> query_embedding) > match_threshold
  order by embedding <=> query_embedding
  limit match_count;
$$;
```

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client
from openai import OpenAI
from pypdf import PdfReader   # pip install pypdf

load_dotenv(override=True)

True

In [3]:
# ── Config ───────────────────────────────────────────────────────────────────
SUPABASE_URL     = "https://ssrxdvbnjfruzikvages.supabase.co"
SUPABASE_API_KEY = os.getenv("SUBABASE_API_KEY")   # matches original .env key name
OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")

PDF_PATH         = Path("raz/employee_handbook_print_1.pdf")  # adjust path if needed
TABLE_NAME       = "hr_documents"                         # new dedicated table
EMBED_MODEL      = "text-embedding-3-small"
CHUNK_SIZE       = 500   # characters per chunk
CHUNK_OVERLAP    = 50    # overlap between consecutive chunks

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH.resolve()}"

In [4]:
# ── Clients ──────────────────────────────────────────────────────────────────
supabase = create_client(SUPABASE_URL, SUPABASE_API_KEY)
openai   = OpenAI(api_key=OPENAI_API_KEY)

In [5]:
# ── Step 1: Extract text from PDF ────────────────────────────────────────────
def extract_pages(pdf_path: Path) -> list[dict]:
    """Return a list of {page_number, text} dicts for every page in the PDF."""
    pages = []
    reader = PdfReader(str(pdf_path))
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()
        if text:
            pages.append({"page_number": i, "text": text})
    print(f"Extracted text from {len(pages)} pages.")
    return pages

pages = extract_pages(PDF_PATH)

Extracted text from 90 pages.


In [6]:
# ── Step 2: Chunk each page ───────────────────────────────────────────────────
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping fixed-size character chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]   # drop empty strings

# Build flat list of {content, page_number, chunk_index}
all_chunks = []
for page in pages:
    for idx, chunk in enumerate(chunk_text(page["text"])):
        all_chunks.append({
            "content":     chunk,
            "page_number": page["page_number"],
            "chunk_index": idx,
        })

print(f"Total chunks to embed: {len(all_chunks)}")

Total chunks to embed: 830


In [7]:
# ── Step 3: Embed & upload ────────────────────────────────────────────────────
def get_embedding(text: str) -> list[float]:
    """Return the embedding vector for a single piece of text."""
    response = openai.embeddings.create(model=EMBED_MODEL, input=text)
    return response.data[0].embedding


def upload_chunks(chunks: list[dict], batch_size: int = 20) -> None:
    """
    Embed each chunk and upsert into Supabase in batches.
    A small sleep between batches avoids OpenAI rate-limit errors.
    """
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i : i + batch_size]
        rows  = []
        for chunk in batch:
            embedding = get_embedding(chunk["content"])
            rows.append({
                "content":     chunk["content"],
                "page_number": chunk["page_number"],
                "chunk_index": chunk["chunk_index"],
                "embedding":   embedding,
            })

        supabase.table(TABLE_NAME).insert(rows).execute()
        print(f"  Uploaded chunks {i+1}–{min(i+batch_size, total)} of {total}")
        time.sleep(0.5)   # be kind to the OpenAI rate limiter


print("Starting embedding + upload …")
upload_chunks(all_chunks)
print("Done — all chunks stored in", TABLE_NAME)

Starting embedding + upload …
  Uploaded chunks 1–20 of 830
  Uploaded chunks 21–40 of 830
  Uploaded chunks 41–60 of 830
  Uploaded chunks 61–80 of 830
  Uploaded chunks 81–100 of 830
  Uploaded chunks 101–120 of 830
  Uploaded chunks 121–140 of 830
  Uploaded chunks 141–160 of 830
  Uploaded chunks 161–180 of 830
  Uploaded chunks 181–200 of 830
  Uploaded chunks 201–220 of 830
  Uploaded chunks 221–240 of 830
  Uploaded chunks 241–260 of 830
  Uploaded chunks 261–280 of 830
  Uploaded chunks 281–300 of 830
  Uploaded chunks 301–320 of 830
  Uploaded chunks 321–340 of 830
  Uploaded chunks 341–360 of 830
  Uploaded chunks 361–380 of 830
  Uploaded chunks 381–400 of 830
  Uploaded chunks 401–420 of 830
  Uploaded chunks 421–440 of 830
  Uploaded chunks 441–460 of 830
  Uploaded chunks 461–480 of 830
  Uploaded chunks 481–500 of 830
  Uploaded chunks 501–520 of 830
  Uploaded chunks 521–540 of 830
  Uploaded chunks 541–560 of 830
  Uploaded chunks 561–580 of 830
  Uploaded chunks 581–6

In [8]:
# ── Step 4: Semantic search helper ───────────────────────────────────────────
def search_handbook(query: str, threshold: float = 0.5,
                    top_k: int = 5) -> list[dict]:
    """Return the most relevant HR handbook passages for a query."""
    query_embedding = get_embedding(query)
    result = supabase.rpc("match_hr_documents", {
        "query_embedding": query_embedding,
        "match_threshold":  threshold,
        "match_count":      top_k,
    }).execute()
    return result.data

In [9]:
# ── Step 5: Example queries ───────────────────────────────────────────────────

# Query 1 — Board meeting public participation
results = search_handbook("How do I address the board of education?")
print("Query: How do I address the board of education?")
for r in results:
    print(f"  [Page {r['page_number']}, chunk {r['chunk_index']},"
          f" sim={r['similarity']:.3f}]")
    print("  ", r["content"][:200], "…\n")

Query: How do I address the board of education?
  [Page 7, chunk 2, sim=0.626]
   igned into Law 
April 25, 2006 
 
BOARD OF EDUCATION MEETINGS 
PUBLIC PARTICIPATION 
 
In accordance with the policy of the board of educa tion, the following regulation shall govern visitors attendin …

  [Page 7, chunk 3, sim=0.535]
   t, and what is expected from the board.  
The letter must be received by the superintendent at least 72 hours prior to the next regularly scheduled meeting in order to be placed 
on the agenda.  (The  …

  [Page 71, chunk 8, sim=0.526]
   Therefore, whenever a complaint is made directly to the board as a whole or to a board member as an 
individual, it will promptly be referred to the school administration for study and possible soluti …

  [Page 72, chunk 0, sim=0.520]
   If all other remedies have been exhausted and a complaint cannot be satisfactorily resolved, the complaint may be appealed to 
the board of education.  No appeal will be heard by the board and no char …

In [10]:
# Query 2 — Payroll deductions
results = search_handbook("Can I stop payroll deductions for my union?")
print("Query: Can I stop payroll deductions for my union?")
for r in results:
    print(f"  [Page {r['page_number']}, chunk {r['chunk_index']},"
          f" sim={r['similarity']:.3f}]")
    print("  ", r["content"][:200], "…\n")

Query: Can I stop payroll deductions for my union?
  [Page 7, chunk 6, sim=0.534]
   fteen (15) business days of receipt of a request, the district shall notify the professional organization of the initiation or termination 
of payroll deductions.  If the request is to terminate a ded …

  [Page 7, chunk 5, sim=0.532]
   authorization of the employee. 
 
However, a school employee may request in writing at any time for the district to immediately terminate or initiate payroll deductions 
to a professional organization …

